In [1]:
# ==========================================
# Notebook 08
# Personalized Candidate Retrieval
# ==========================================

import pandas as pd
import numpy as np

import faiss

In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [3]:
interactions_df = pd.read_csv("../data/user_interactions.csv")

interactions_df.head()

,user_id,item_id,rating
0,101,1,5
1,101,2,5
2,101,5,4
3,102,1,5
4,102,8,5


In [4]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [5]:
user_embeddings = np.load("../data/user_embeddings.npy")

user_embeddings.shape

(5, 384)

In [6]:
index = faiss.read_index("../data/item_index.faiss")

index.ntotal

0

In [7]:
print("Items:", len(profiles_df))

print()

print("Users:", interactions_df["user_id"].nunique())

print()

print("Embeddings:", item_embeddings.shape)

Items: 10

Users: 5

Embeddings: (10, 384)


In [8]:
user_ids = sorted(interactions_df["user_id"].unique())

user_ids

[101, 102, 103, 104, 105]

In [9]:
user_mapping = {user_id: idx for idx, user_id in enumerate(user_ids)}

user_mapping

{101: 0, 102: 1, 103: 2, 104: 3, 105: 4}

In [10]:
def retrieve_candidates(user_id, top_k=10):

    if user_id not in user_mapping:

        return pd.DataFrame()

    user_index = user_mapping[user_id]

    user_vector = user_embeddings[user_index].astype("float32")

    distances, indices = index.search(user_vector.reshape(1, -1), top_k)

    recommendations = []

    for rank, idx in enumerate(indices[0]):

        recommendations.append(
            {
                "rank": rank + 1,
                "item_id": profiles_df.iloc[idx]["item_id"],
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
                "distance": float(distances[0][rank]),
            }
        )

    return pd.DataFrame(recommendations)

In [11]:
retrieve_candidates(101)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38
5,6,10,Large Language Models,Artificial Intelligence,3.402823e+38
6,7,10,Large Language Models,Artificial Intelligence,3.402823e+38
7,8,10,Large Language Models,Artificial Intelligence,3.402823e+38
8,9,10,Large Language Models,Artificial Intelligence,3.402823e+38
9,10,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [12]:
retrieve_candidates(102)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38
5,6,10,Large Language Models,Artificial Intelligence,3.402823e+38
6,7,10,Large Language Models,Artificial Intelligence,3.402823e+38
7,8,10,Large Language Models,Artificial Intelligence,3.402823e+38
8,9,10,Large Language Models,Artificial Intelligence,3.402823e+38
9,10,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [13]:
retrieve_candidates(104)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38
5,6,10,Large Language Models,Artificial Intelligence,3.402823e+38
6,7,10,Large Language Models,Artificial Intelligence,3.402823e+38
7,8,10,Large Language Models,Artificial Intelligence,3.402823e+38
8,9,10,Large Language Models,Artificial Intelligence,3.402823e+38
9,10,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [14]:
def get_user_history(user_id):

    history = interactions_df[interactions_df["user_id"] == user_id]

    return pd.merge(
        history, profiles_df[["item_id", "title", "category"]], on="item_id"
    )

In [15]:
get_user_history(101)

,user_id,item_id,rating,title,category
0,101,1,5,The Early Days of Stripe,Startup
1,101,2,5,Building SpaceX,Startup
2,101,5,4,Cloud Computing Fundamentals,Technology


In [16]:
user_id = 101

print("USER HISTORY")

display(get_user_history(user_id))

print()

print("RECOMMENDATIONS")

display(retrieve_candidates(user_id))

USER HISTORY


,user_id,item_id,rating,title,category
0,101,1,5,The Early Days of Stripe,Startup
1,101,2,5,Building SpaceX,Startup
2,101,5,4,Cloud Computing Fundamentals,Technology



RECOMMENDATIONS


,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38
5,6,10,Large Language Models,Artificial Intelligence,3.402823e+38
6,7,10,Large Language Models,Artificial Intelligence,3.402823e+38
7,8,10,Large Language Models,Artificial Intelligence,3.402823e+38
8,9,10,Large Language Models,Artificial Intelligence,3.402823e+38
9,10,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [17]:
def retrieve_new_candidates(user_id, top_k=10):

    if user_id not in user_mapping:

        return pd.DataFrame()

    consumed_items = interactions_df[interactions_df["user_id"] == user_id][
        "item_id"
    ].tolist()

    user_index = user_mapping[user_id]

    user_vector = user_embeddings[user_index].astype("float32")

    distances, indices = index.search(user_vector.reshape(1, -1), 50)

    recommendations = []

    rank = 1

    for idx in indices[0]:

        item_id = profiles_df.iloc[idx]["item_id"]

        if item_id in consumed_items:

            continue

        recommendations.append(
            {
                "rank": rank,
                "item_id": item_id,
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
            }
        )

        rank += 1

        if len(recommendations) >= top_k:

            break

    return pd.DataFrame(recommendations)

In [18]:
retrieve_new_candidates(101)

,rank,item_id,title,category
0,1,10,Large Language Models,Artificial Intelligence
1,2,10,Large Language Models,Artificial Intelligence
2,3,10,Large Language Models,Artificial Intelligence
3,4,10,Large Language Models,Artificial Intelligence
4,5,10,Large Language Models,Artificial Intelligence
5,6,10,Large Language Models,Artificial Intelligence
6,7,10,Large Language Models,Artificial Intelligence
7,8,10,Large Language Models,Artificial Intelligence
8,9,10,Large Language Models,Artificial Intelligence
9,10,10,Large Language Models,Artificial Intelligence


In [19]:
retrieve_new_candidates(102)

,rank,item_id,title,category
0,1,10,Large Language Models,Artificial Intelligence
1,2,10,Large Language Models,Artificial Intelligence
2,3,10,Large Language Models,Artificial Intelligence
3,4,10,Large Language Models,Artificial Intelligence
4,5,10,Large Language Models,Artificial Intelligence
5,6,10,Large Language Models,Artificial Intelligence
6,7,10,Large Language Models,Artificial Intelligence
7,8,10,Large Language Models,Artificial Intelligence
8,9,10,Large Language Models,Artificial Intelligence
9,10,10,Large Language Models,Artificial Intelligence


In [20]:
retrieve_new_candidates(104)

""


In [21]:
candidate_results = []

In [22]:
for user_id in user_ids:

    candidates = retrieve_new_candidates(user_id, top_k=5)

    for _, row in candidates.iterrows():

        candidate_results.append(
            {
                "user_id": user_id,
                "item_id": row["item_id"],
                "title": row["title"],
                "category": row["category"],
                "rank": row["rank"],
            }
        )

In [23]:
candidate_df = pd.DataFrame(candidate_results)

candidate_df.head()

,user_id,item_id,title,category,rank
0,101,10,Large Language Models,Artificial Intelligence,1
1,101,10,Large Language Models,Artificial Intelligence,2
2,101,10,Large Language Models,Artificial Intelligence,3
3,101,10,Large Language Models,Artificial Intelligence,4
4,101,10,Large Language Models,Artificial Intelligence,5


In [24]:
candidate_df["item_id"].nunique()

1

In [25]:
candidate_df.groupby("user_id").size()

user_id
101    5
102    5
103    5
105    5
dtype: int64

In [26]:
candidate_df.describe(include="all")

,user_id,item_id,title,category,rank
count,20.000000,20.0,20,20,20.000000
unique,NaN,NaN,1,1,NaN
top,NaN,NaN,Large Language Models,Artificial Intelligence,NaN
freq,NaN,NaN,20,20,NaN
mean,102.750000,10.0,NaN,NaN,3.000000
std,1.517442,0.0,NaN,NaN,1.450953
min,101.000000,10.0,NaN,NaN,1.000000
25%,101.750000,10.0,NaN,NaN,2.000000
50%,102.500000,10.0,NaN,NaN,3.000000
75%,103.500000,10.0,NaN,NaN,4.000000


In [27]:
candidate_df.to_csv("../data/candidate_recommendations.csv", index=False)

In [31]:
saved_df = pd.read_csv("../data/candidate_recommendations.csv")

saved_df

,user_id,item_id,title,category,rank
0,101,10,Large Language Models,Artificial Intelligence,1
1,101,10,Large Language Models,Artificial Intelligence,2
2,101,10,Large Language Models,Artificial Intelligence,3
3,101,10,Large Language Models,Artificial Intelligence,4
4,101,10,Large Language Models,Artificial Intelligence,5
5,102,10,Large Language Models,Artificial Intelligence,1
6,102,10,Large Language Models,Artificial Intelligence,2
7,102,10,Large Language Models,Artificial Intelligence,3
8,102,10,Large Language Models,Artificial Intelligence,4
9,102,10,Large Language Models,Artificial Intelligence,5
